In [ ]:
import pandas as pd
import numpy as np

# ─── Load Data ───────────────────────────────────────────────
excel_file = "NovaPay_FPA_Model_FIXED_US_DATES.xlsx"

revenue   = pd.read_excel(excel_file, sheet_name="Revenue_Data")
expenses  = pd.read_excel(excel_file, sheet_name="Expense_Data")
budget    = pd.read_excel(excel_file, sheet_name="Budget_Data")
kpi       = pd.read_excel(excel_file, sheet_name="KPI_Data")
scenarios = pd.read_excel(excel_file, sheet_name="Scenario_Inputs")
summary   = pd.read_excel(excel_file, sheet_name="Monthly_Summary")

for df in [revenue, expenses, budget, kpi, summary]:
    df["Month_Label"]  = df["Month_Label"].astype(str)
    df["Month_Number"] = df["Month_Number"].astype(int)

print("Files loaded successfully.")
print(f"Revenue rows: {len(revenue)}")
print(f"Expense rows: {len(expenses)}")
print(f"KPI rows:     {len(kpi)}")


# ─── Monthly Revenue by Stream ───────────────────────────────
monthly_revenue = revenue.groupby(
    ["Month_Number", "Month_Label"], as_index=False
).agg(
    Subscription_Revenue      = ("Subscription_Revenue",      "sum"),
    Transaction_Fee_Revenue   = ("Transaction_Fee_Revenue",   "sum"),
    Premium_Analytics_Revenue = ("Premium_Analytics_Revenue", "sum"),
    Setup_Fee_Revenue         = ("Setup_Fee_Revenue",         "sum"),
    Total_Revenue             = ("Total_Revenue",             "sum")
).sort_values("Month_Number").reset_index(drop=True)

print("\n=== Monthly Revenue by Stream ===")
print(monthly_revenue.to_string(index=False))


# ─── Expense Variance with Overrun Flag ──────────────────────
expense_variance = expenses.copy()
expense_variance["Expense_Variance"]   = expense_variance["Actual_Expense"] - expense_variance["Budget_Expense"]
expense_variance["Expense_Variance_%"] = expense_variance["Expense_Variance"] / expense_variance["Budget_Expense"]
expense_variance["Variance_Flag"] = np.where(
    expense_variance["Expense_Variance_%"] >= 0.10, "High Overrun",
    np.where(expense_variance["Expense_Variance_%"] > 0, "Moderate Overrun", "Within Budget")
)

print("\n=== Top 10 Expense Overruns ===")
print(expense_variance[
    ["Month_Label", "Department", "Expense_Category",
     "Vendor", "Budget_Expense", "Actual_Expense",
     "Expense_Variance", "Expense_Variance_%", "Variance_Flag"]
].sort_values("Expense_Variance", ascending=False).head(10).to_string(index=False))


# ─── KPI Risk Flags ──────────────────────────────────────────
kpi_risk = kpi.copy()
kpi_risk["Churn_Risk"] = np.where(
    kpi_risk["Churn_Rate"] >= 0.08, "High Churn Risk",
    np.where(kpi_risk["Churn_Rate"] >= 0.04, "Moderate Churn Risk", "Low Churn Risk")
)
kpi_risk["CAC_Risk"] = np.where(
    kpi_risk["CAC"] >= 10000, "High CAC",
    np.where(kpi_risk["CAC"] >= 5000, "Moderate CAC", "Low CAC")
)

print("\n=== KPI Risk Flags ===")
print(kpi_risk[[
    "Month_Label", "Active_Customers", "Churn_Rate",
    "CAC", "Net_Revenue_Retention", "Churn_Risk", "CAC_Risk"
]].to_string(index=False))


# ─── Monthly EBITDA Analysis ─────────────────────────────────
ebitda_analysis = summary[[
    "Month_Number", "Month_Label",
    "Actual_Revenue", "Actual_Expenses", "Actual_COGS",
    "Gross_Profit", "Gross_Margin", "EBITDA", "EBITDA_Margin"
]].sort_values("Month_Number").reset_index(drop=True)
ebitda_analysis["EBITDA_Margin_%"] = (ebitda_analysis["EBITDA_Margin"].round(4) * 100).round(2)

print("\n=== Monthly EBITDA Analysis ===")
print(ebitda_analysis.drop(columns=["EBITDA_Margin"]).to_string(index=False))


# ─── Scenario Forecast ───────────────────────────────────────
latest_revenue  = monthly_revenue.sort_values("Month_Number").iloc[-1]["Total_Revenue"]
latest_expenses = summary.sort_values("Month_Number").iloc[-1]["Actual_Expenses"]
latest_cogs     = summary.sort_values("Month_Number").iloc[-1]["Actual_COGS"]

forecast = scenarios.copy()
forecast["Forecast_Revenue"]  = round(latest_revenue  * (1 + forecast["Customer_Growth_Rate"]) * (1 - forecast["Churn_Rate"]), 2)
forecast["Forecast_Expenses"] = round(latest_expenses * (1 + forecast["Payroll_Inflation"]) * (1 + forecast["Marketing_Spend_Change"]), 2)
forecast["Forecast_EBITDA"]   = round(forecast["Forecast_Revenue"] - latest_cogs - forecast["Forecast_Expenses"], 2)

print("\n=== Scenario Forecast ===")
print(forecast[[
    "Scenario", "Customer_Growth_Rate", "Churn_Rate",
    "CAC_Change", "Payroll_Inflation",
    "Forecast_Revenue", "Forecast_Expenses", "Forecast_EBITDA"
]].to_string(index=False))


# ─── CFO Commentary ──────────────────────────────────────────
latest_kpi    = kpi.sort_values("Month_Number").iloc[-1]
latest_rev    = monthly_revenue.sort_values("Month_Number").iloc[-1]["Total_Revenue"]
latest_exp    = summary.sort_values("Month_Number").iloc[-1]["Actual_Expenses"]
latest_ebitda = summary.sort_values("Month_Number").iloc[-1]["EBITDA"]
prev_ebitda   = summary.sort_values("Month_Number").iloc[-2]["EBITDA"]
latest_month  = kpi.sort_values("Month_Number").iloc[-1]["Month_Label"]

commentary = []
commentary.append(f"\n=== NovaPay CFO Commentary — {latest_month} ===\n")
commentary.append(f"1. Latest monthly revenue: ${latest_rev:,.2f}")
commentary.append(f"2. Latest monthly operating expenses: ${latest_exp:,.2f}")
commentary.append(f"3. EBITDA: ${latest_ebitda:,.2f} ({'improving' if latest_ebitda > prev_ebitda else 'declining'} vs prior month)")

if latest_kpi["Churn_Rate"] >= 0.08:
    commentary.append("4. Churn is a HIGH RISK area — immediate retention action required.")
elif latest_kpi["Churn_Rate"] >= 0.04:
    commentary.append("4. Churn is moderate — monitor closely.")
else:
    commentary.append("4. Churn is within a manageable range.")

if latest_kpi["CAC"] >= 10000:
    commentary.append("5. Customer acquisition cost is HIGH — review marketing efficiency.")
elif latest_kpi["CAC"] >= 5000:
    commentary.append("5. Customer acquisition cost is moderate — manageable for now.")
else:
    commentary.append("5. Customer acquisition cost is low — healthy.")

commentary.append("\nRecommended Actions:")
commentary.append("  → Control variable costs: Marketing Spend and Cloud Hosting are the biggest overrun areas.")
commentary.append("  → Monitor June churn closely — first churn event in the dataset.")
commentary.append("  → Push Basic plan customers toward Premium to improve revenue quality.")
commentary.append("  → Track CAC vs LTV ratio as customer base grows.")

for line in commentary:
    print(line)